In [63]:
import pandas as pd
import requests
import yfinance as yf
from datetime import datetime
import holidays

#f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial={date_init}&@dataFinalCotacao={date_final}&$top=10000&$format=json&$select=cotacaoCompra,dataHoraCotacao"

## Creating csv dolar

In [ ]:
#Dont using 
actual_date = datetime.today().strftime('%d/%m/%Y')

datas = [
    "10/05/2004",
    "10/05/2014",
    "10/05/2024",
    actual_date
]
df = []
# Loop de 2 em 2
for i in range(0, len(datas), 2):
    d1 = datas[i]
    d2 = datas[i+1]
    print(d1,d2)
    data = requests.get(f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.1/dados?formato=json&dataInicial={d1}&dataFinal={d2}")
    df.append(data.json())

10/05/2004 10/05/2014


In [ ]:
# doc api: https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/aplicacao#!/recursos/CotacaoDolarPeriodo#eyJmb3JtdWxhcmlvIjp7IiRmb3JtYXQiOiJqc29uIiwiJHRvcCI6MTAwMDAsImRhdGFJbmljaWFsIjoiMDEtMDEtMjAwNCIsImRhdGFGaW5hbENvdGFjYW8iOiIwMS0wMS0yMDEzIn0sInByb3ByaWVkYWRlcyI6WzEsMl19
actual_date = datetime.today().strftime('%m-%d-%Y')
#filtering date in 10year in 10
datas = [
    '05-10-2004',
    '05-10-2014',
    '05-10-2024',
    actual_date
]
# purchase dolar
values_dolar_purchase = []
dates_dolar_purchase = []
# sale dolar
values_dolar_sale = []
dates_dolar_sale = []
for buy_sale in ["cotacaoCompra","cotacaoVenda"]:
    # Loop by 2 in 2 by date
    for i in range(0, len(datas), 2):
        date_init = datas[i]
        date_final = datas[i+1]
        print(date_init,date_final)  
        resp = requests.get(f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,\
                            dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{date_init}'&@dataFinalCotacao='{date_final}'&$top=10000&$\
                            format=json&$select={buy_sale},dataHoraCotacao") # cotacaoCompra
        data = resp.json()
        match buy_sale:
            case "cotacaoCompra":
                [values_dolar_purchase.append(data["value"][i]["cotacaoCompra"]) for i in range(len(data["value"]))]
                [dates_dolar_purchase.append(data["value"][i]["dataHoraCotacao"]) for i in range(len(data["value"]))]
            case "cotacaoVenda":
                [values_dolar_sale.append(data["value"][i]["cotacaoVenda"]) for i in range(len(data["value"]))]
                [dates_dolar_sale.append(data["value"][i]["dataHoraCotacao"]) for i in range(len(data["value"]))]

05-10-2004 05-10-2014
05-10-2024 12-19-2025
05-10-2004 05-10-2014
05-10-2024 12-19-2025


In [26]:
dolar = {"Valor Venda Dolar": values_dolar_sale, 
        "Valor Compra Dolar": values_dolar_purchase, 
        "Data Compra Venda Dolar": dates_dolar_purchase}
df_dolar = pd.DataFrame(dolar)
df_dolar["Data Compra Venda Dolar"] = pd.to_datetime(df_dolar["Data Compra Venda Dolar"]).dt.strftime('%Y-%m-%d')
df_dolar["Valor Venda Dolar"] = df_dolar["Valor Venda Dolar"].astype(float)
df_dolar["Valor Compra Dolar"] = df_dolar["Valor Compra Dolar"].astype(float)
# transform date in index and drop 
df_dolar.index = df_dolar["Data Compra Venda Dolar"]
df_dolar = df_dolar.drop(columns=["Data Compra Venda Dolar"])

In [29]:
df_dolar.to_csv("..\data\dolar.csv")#data turn index

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ferna\AppData\Local\Temp\ipykernel_43760\1183323993.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_dolar.to_csv("..\data\dolar.csv")#data turn index


## Getting Brent

In [ ]:
# Define o ticker do Brent (BZ=F)
ticker = "BZ=F"

# Define as datas (do início de 2004 até a data de hoje)
data_inicio = "2004-01-01"
data_fim = datetime.today().strftime('%Y-%m-%d')

print(f"Baixando dados do Brent ({ticker}) de {data_inicio} até {data_fim}...")

# Baixa os dados
# auto_adjust=True ajuda a corrigir distorções de contratos antigos
dados = yf.download(ticker, start=data_inicio, end=data_fim, auto_adjust=True)

if not dados.empty:
    print(f"\nSucesso! Foram baixados {len(dados)} dias de negociação.")
    
    # Exibe os 5 primeiros registros (2004/2007)
    print("\n--- Primeiros Registros ---")
    print(dados.head())

    # Exibe os 5 últimos registros (Hoje)
    print("\n--- Últimos Registros ---")
    print(dados.tail())
    
    # Se quiser salvar em Excel para conferir:
    # dados.to_excel("brent_historico.xlsx")
else:
    print("Não foi possível encontrar dados para este período. O Yahoo pode ter mudado o ticker.")

## Merge Dolar csv with ANP parquet

In [3]:
df_dolar = pd.read_csv(r"..\data\dolar.csv", sep=",", index_col=0)
df = pd.read_parquet(r"..\data\anp_parquet.parquet")
df["Data da Coleta"] = pd.to_datetime(df["Data da Coleta"])
df = df.drop(columns=["Regiao - Sigla", "Nome da Rua","Numero Rua", "Complemento", "Bairro", "Regiao - Sigla.1"])
# drop values nan from columns Bandeira and Unidade de Medida
df = df.dropna(subset=["Bandeira","Unidade de Medida"])
df["Unidade de Medida"] = df["Unidade de Medida"].replace("R$ / mÂ³","R$ / m³")


In [4]:
date_init = '2004-05-10'
date_final = '2025-12-05'
all_dates = pd.date_range(date_init, date_final, freq='D')
df_calendar = pd.DataFrame(all_dates, columns=['data'])


In [5]:
df_calendar["is_weekend"] = df_calendar["data"].dt.weekday >= 5
br_feriados = holidays.Brazil(years=range(2004, 2026))
# Verifica se cada data no DataFrame é um feriado
df_calendar['is_holiday'] = df_calendar['data'].isin(br_feriados)

# 5. Criar uma coluna que marca qualquer dia não útil
df_calendar['day_no_util'] = df_calendar['is_weekend'] | df_calendar['is_holiday']
dias_nao_uteis = df_calendar[df_calendar['day_no_util']].copy()
print(f"Total de dias não úteis no período: {len(dias_nao_uteis)}")
print(dias_nao_uteis.head(10))

Total de dias não úteis no período: 2395
         data  is_weekend  is_holiday  day_no_util
5  2004-05-15        True       False         True
6  2004-05-16        True       False         True
12 2004-05-22        True       False         True
13 2004-05-23        True       False         True
19 2004-05-29        True       False         True
20 2004-05-30        True       False         True
26 2004-06-05        True       False         True
27 2004-06-06        True       False         True
33 2004-06-12        True       False         True
34 2004-06-13        True       False         True


C:\Users\ferna\AppData\Local\Temp\ipykernel_19704\624280955.py:4: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df_calendar['is_holiday'] = df_calendar['data'].isin(br_feriados)


In [45]:
df_calendar[df_calendar["is_holiday"] == True]#["data"]#[df_calendar["is_holiday"] == True]["data"].max()

,data,is_weekend,is_holiday,day_no_util
120,2004-09-07,False,True,True
155,2004-10-12,False,True,True
176,2004-11-02,False,True,True
189,2004-11-15,False,True,True
229,2004-12-25,True,True,True
...,...,...,...,...
7790,2025-09-07,True,True,True
7825,2025-10-12,True,True,True
7846,2025-11-02,True,True,True
7859,2025-11-15,True,True,True


In [30]:
df_dolar[df_dolar.index.duplicated() == True]

,Valor Venda Dolar,Valor Compra Dolar
Data Compra Venda Dolar,,
2025-04-23,5.688,5.6874


In [59]:

# Caso tenha valores nan, drop em 1 e mantenha o primeiro registro do index
df_dolar = df_dolar[~df_dolar.index.duplicated(keep='first')]


df_dolar.index = pd.to_datetime(df_dolar.index)

# Reindexar o DataFrame para incluir TODOS os dias do período
# Isso criará linhas com valores NaN para os dias que estavam faltando (não úteis)
todas_as_datas_serie = pd.date_range(start=df_dolar.index.min(), end=df_dolar.index.max(), freq='D')

dolar_completo = df_dolar.reindex(todas_as_datas_serie)

# Preencher os valores faltantes (os dias não úteis) com o último valor válido
# Forward fill - preenche NaN com último valor válido anterior
dolar_completo['Valor Venda Dolar'] = dolar_completo['Valor Venda Dolar'].ffill()
dolar_completo['Valor Compra Dolar'] = dolar_completo['Valor Compra Dolar'].ffill()

dolar_completo.index = dolar_completo.index.rename("Data Coleta Dolar")
# dolar_completo

In [60]:
df_merged = df.merge(
    dolar_completo, 
    how="inner", 
    left_on="Data da Coleta", 
    right_index=True
)

In [61]:
df_merged

,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira,Valor Venda Dolar,Valor Compra Dolar
0,CE,SOBRAL,AUTO POSTO APRAZIVEL LTDA,00.422.849/0001-53,00000-000,DIESEL,2004-05-10,1.470,1.507225,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.,3.1249,3.1241
1,BA,JAGUAQUARA,LUZITALIA COM. E TRANSP. DE DERIVADOS DE PETRO...,01.073.545/0001-90,00000-000,DIESEL,2004-05-10,1.229,1.153090,R$ / litro,COSAN LUBRIFICANTES,3.1249,3.1241
2,GO,APARECIDA DE GOIANIA,DPL DISTRIBUIDORA DE PETROLEO LTDA,01.157.392/0001-60,00000-000,DIESEL,2004-05-10,1.399,1.250000,R$ / litro,BRANCA,3.1249,3.1241
3,GO,RIO VERDE,POSTO MORADA DO SOL LTDA,01.477.306/0001-04,00000-000,DIESEL,2004-05-10,1.540,1.300010,R$ / litro,LIQUIGÃS,3.1249,3.1241
4,SC,BALNEARIO CAMBORIU,VENTURELLA & SOUZA LTDA,02.713.999/0001-41,00000-000,DIESEL,2004-05-10,1.379,1.339000,R$ / litro,RAIZEN,3.1249,3.1241
...,...,...,...,...,...,...,...,...,...,...,...,...,...
22497208,SP,GUARULHOS,GRACANDU AUTO POSTO LTDA,06.261.544/0001-93,07144-000,GASOLINA ADITIVADA,2025-07-05,5.690,NaN,R$ / litro,BRANCA,5.4090,5.4084
22497209,SP,GUARULHOS,AUTO POSTO VENETO LTDA,01.791.354/0001-64,07170-350,GASOLINA ADITIVADA,2025-07-05,6.790,NaN,R$ / litro,ALE,5.4090,5.4084
22497210,SP,GUARULHOS,EMPRESA 4B DE SERVICOS AUTOMOTIVOS LTDA,00.541.478/0001-29,07190-001,GASOLINA ADITIVADA,2025-07-05,5.790,NaN,R$ / litro,IPIRANGA,5.4090,5.4084
22497211,SP,GUARULHOS,NIKIGAS COMERCIAL LTDA,04.438.260/0003-66,07013-100,GNV,2025-07-05,4.590,NaN,R$ / m³,BRANCA,5.4090,5.4084


In [62]:
df_merged.to_parquet(r"..\data\anp_dolar.parquet")

# Extracting IPCA, SELIC & IBC-Br

In [107]:
codes = {
    'ipca': 433,
    'selic': 11,
    'pib_mensal': 24363#24369,
}
actual_date = datetime.today().strftime('%d/%m/%Y')
dates = [
    '10/05/2004',
    '10/05/2014',
    '10/05/2024',
    actual_date
]
c = 0
data_collected = {
    433: [],
    11: [],
    24363: []
}
for i in range(0, len(dates), 2):
    for code in codes:
        url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codes[code]}/dados?formato=json&dataInicial={dates[i]}&dataFinal={dates[i+1]}"
        resp = requests.get(url)
        print(url)
        print(f"{codes[code]}:{resp.json()}")
        data_collected[codes[code]].append(resp.json())
        c += 1
        if c == 2:
            print("=============")
            c=0

https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=10/05/2004&dataFinal=10/05/2014
433:[{'data': '01/05/2004', 'valor': '0.51'}, {'data': '01/06/2004', 'valor': '0.71'}, {'data': '01/07/2004', 'valor': '0.91'}, {'data': '01/08/2004', 'valor': '0.69'}, {'data': '01/09/2004', 'valor': '0.33'}, {'data': '01/10/2004', 'valor': '0.44'}, {'data': '01/11/2004', 'valor': '0.69'}, {'data': '01/12/2004', 'valor': '0.86'}, {'data': '01/01/2005', 'valor': '0.58'}, {'data': '01/02/2005', 'valor': '0.59'}, {'data': '01/03/2005', 'valor': '0.61'}, {'data': '01/04/2005', 'valor': '0.87'}, {'data': '01/05/2005', 'valor': '0.49'}, {'data': '01/06/2005', 'valor': '-0.02'}, {'data': '01/07/2005', 'valor': '0.25'}, {'data': '01/08/2005', 'valor': '0.17'}, {'data': '01/09/2005', 'valor': '0.35'}, {'data': '01/10/2005', 'valor': '0.75'}, {'data': '01/11/2005', 'valor': '0.55'}, {'data': '01/12/2005', 'valor': '0.36'}, {'data': '01/01/2006', 'valor': '0.59'}, {'data': '01/02/2006

In [108]:
len_datas = 0
for i in data_collected:
    datas = data_collected[i]
    for n in range(len(datas)):
        data = datas[n]
        len_datas += len(data)
        for i2 in data:
            ...# print(f"{i}:{i2}")
print(len_datas)

3225


In [109]:

rows = []

for codigo in data_collected:
    listas = data_collected[codigo]     # lista de listas

    for lista in listas:                # lista interna
        for item in lista:              # dict final
            rows.append({
                "codigo": codigo,
                "data": item.get("data"),
                "valor": float(item.get("valor"))
            })

df = pd.DataFrame(rows)

In [110]:
df = df.replace({433:'ipca',11:'selic',24363:'pib_mensal'})

In [111]:
df

,codigo,data,valor
0,ipca,01/05/2004,0.51000
1,ipca,01/06/2004,0.71000
2,ipca,01/07/2004,0.91000
3,ipca,01/08/2004,0.69000
4,ipca,01/09/2004,0.33000
...,...,...,...
3220,pib_mensal,01/07/2025,113.23431
3221,pib_mensal,01/08/2025,110.48903
3222,pib_mensal,01/09/2025,109.54869
3223,pib_mensal,01/10/2025,109.90366


In [112]:
df["data"] = pd.to_datetime(df["data"], dayfirst=True)

date_spine = pd.DataFrame({
    "data": pd.date_range(
        start=df["data"].min(),
        end=df["data"].max(),
        freq="D"  # daily (use 'MS' to monthly)
    )
})
date_spine

,data
0,2004-05-01
1,2004-05-02
2,2004-05-03
3,2004-05-04
4,2004-05-05
...,...
7933,2026-01-19
7934,2026-01-20
7935,2026-01-21
7936,2026-01-22


In [113]:
df_wide = (
    df.pivot_table(
        index="data",
        columns="codigo",
        values="valor",
        aggfunc="last"
    )
    .reset_index()
)


In [114]:
df_wide = (
    date_spine
    .merge(df_wide, on="data", how="left")
    .sort_values("data")
)
df_wide


,data,ipca,pib_mensal,selic
0,2004-05-01,0.51,74.42369,NaN
1,2004-05-02,NaN,NaN,NaN
2,2004-05-03,NaN,NaN,NaN
3,2004-05-04,NaN,NaN,NaN
4,2004-05-05,NaN,NaN,NaN
...,...,...,...,...
7933,2026-01-19,NaN,NaN,0.055131
7934,2026-01-20,NaN,NaN,0.055131
7935,2026-01-21,NaN,NaN,0.055131
7936,2026-01-22,NaN,NaN,0.055131


In [ ]:
df_wide["ipca"] = df_wide["ipca"].ffill()
df_wide["pib_mensal"] = df_wide["pib_mensal"].ffill()
df_wide["selic"] = df_wide["selic"].ffill() 

In [116]:
df_wide = df_wide[df_wide["data"] >= "2004-05-10"]

In [125]:
dolar

,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira,Valor Venda Dolar,Valor Compra Dolar
0,CE,SOBRAL,AUTO POSTO APRAZIVEL LTDA,00.422.849/0001-53,00000-000,DIESEL,2004-05-10,1.470,1.507225,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.,3.1249,3.1241
1,BA,JAGUAQUARA,LUZITALIA COM. E TRANSP. DE DERIVADOS DE PETRO...,01.073.545/0001-90,00000-000,DIESEL,2004-05-10,1.229,1.153090,R$ / litro,COSAN LUBRIFICANTES,3.1249,3.1241
2,GO,APARECIDA DE GOIANIA,DPL DISTRIBUIDORA DE PETROLEO LTDA,01.157.392/0001-60,00000-000,DIESEL,2004-05-10,1.399,1.250000,R$ / litro,BRANCA,3.1249,3.1241
3,GO,RIO VERDE,POSTO MORADA DO SOL LTDA,01.477.306/0001-04,00000-000,DIESEL,2004-05-10,1.540,1.300010,R$ / litro,LIQUIGÃS,3.1249,3.1241
4,SC,BALNEARIO CAMBORIU,VENTURELLA & SOUZA LTDA,02.713.999/0001-41,00000-000,DIESEL,2004-05-10,1.379,1.339000,R$ / litro,RAIZEN,3.1249,3.1241
...,...,...,...,...,...,...,...,...,...,...,...,...,...
22497208,SP,GUARULHOS,GRACANDU AUTO POSTO LTDA,06.261.544/0001-93,07144-000,GASOLINA ADITIVADA,2025-07-05,5.690,NaN,R$ / litro,BRANCA,5.4090,5.4084
22497209,SP,GUARULHOS,AUTO POSTO VENETO LTDA,01.791.354/0001-64,07170-350,GASOLINA ADITIVADA,2025-07-05,6.790,NaN,R$ / litro,ALE,5.4090,5.4084
22497210,SP,GUARULHOS,EMPRESA 4B DE SERVICOS AUTOMOTIVOS LTDA,00.541.478/0001-29,07190-001,GASOLINA ADITIVADA,2025-07-05,5.790,NaN,R$ / litro,IPIRANGA,5.4090,5.4084
22497211,SP,GUARULHOS,NIKIGAS COMERCIAL LTDA,04.438.260/0003-66,07013-100,GNV,2025-07-05,4.590,NaN,R$ / m³,BRANCA,5.4090,5.4084


In [130]:
dolar = pd.read_parquet(r"..\data\anp_dolar.parquet")

df_merged_dolar_selic = dolar.merge(
    df_wide, 
    how="inner", 
    left_on="Data da Coleta", 
    right_on="data"
    )

In [135]:
df_merged_dolar_selic = df_merged_dolar_selic.drop(columns=["data"])

In [137]:
df_merged_dolar_selic.to_parquet(r"..\data\anp_dolar_enriched.parquet")